# Ollama + Cloudflare Quick Tunnel (Google Colab)

Runs an Ollama server inside Colab and exposes it publicly through a Cloudflare quick tunnel.
Run the cells top to bottom.

In [ ]:
# Install cloudflared from the official GitHub release
!rm -f cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared --version

In [ ]:
# Install Ollama (no-op if it is already installed)
!command -v ollama >/dev/null 2>&1 || curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

In [ ]:
import os
import subprocess
import time
import urllib.request

# Stop any Ollama process left over from a previous run
subprocess.run(["pkill", "-9", "-f", "ollama"], capture_output=True)
time.sleep(2)

# Start the server so the tunnel can reach it (bind 0.0.0.0, allow all origins)
with open("ollama.log", "w") as log:
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=log,
        stderr=log,
        env=dict(os.environ, OLLAMA_HOST="0.0.0.0", OLLAMA_ORIGINS="*"),
        start_new_session=True,
    )

# Wait (up to 30s) for the server to answer before continuing
for _ in range(30):
    try:
        with urllib.request.urlopen("http://127.0.0.1:11434", timeout=2) as r:
            print("Ollama is up:", r.read().decode())
        break
    except Exception:
        time.sleep(1)
else:
    print("Ollama failed to start. Log tail:")
    !tail -n 20 ollama.log

In [ ]:
MODEL = "deepseek-r1:14b"  # swap for e.g. "gemma3:12b" or "llama3.1:8b , gemma4:12b, jaahas/qwen3.5-uncensored:9b"
# fredrezones55/Qwen3.5-Uncensored-HauhauCS-Aggressive:9b, jaahas/qwen3.5-uncensored:9b, orcarouter/Qwen3.8-27B-Uncensored:q8_0

!ollama pull {MODEL}
!ollama list

In [ ]:
import re
import subprocess
import time

# Start a Cloudflare quick tunnel in the background
with open("cloudflared.log", "w") as log:
    subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:11434"],
        stdout=log,
        stderr=log,
        start_new_session=True,
    )

# Wait (up to 30s) for the public URL to appear in the log
url = None
for _ in range(30):
    time.sleep(1)
    matches = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open("cloudflared.log").read())
    if matches:
        url = matches[0]
        break

if url:
    print("Public URL:      ", url)
    print("OpenAI endpoint: ", f"{url}/v1/chat/completions")
else:
    print("Tunnel failed to start. Log tail:")
    !tail -n 20 cloudflared.log

## Notes

- The tunnel URL changes on every run and only lives while the session is running.
- Ollama exposes an OpenAI-compatible API at `/v1`, so any API key works with it.
- Stop the tunnel with `!pkill -f cloudflared`.